# **BIG DATA K08 - Week 6 (spark-mllib)**

---

*(Link Kaggle :
https://www.kaggle.com/datasets/thuandao/online-shoppers-purchasing-dataset)*

## Latar Belakang
Perkembangan e-commerce yang pesat membuat perusahaan harus mampu memahami perilaku pengguna secara mendalam. Tidak semua pengunjung yang datang ke sebuah website akan melakukan pembelian. Oleh karena itu, diperlukan analisis data untuk mengetahui faktor-faktor yang mempengaruhi keputusan pembelian pengguna.

Dataset ini merepresentasikan aktivitas pengguna pada sebuah website e-commerce, seperti jumlah halaman yang dikunjungi, durasi kunjungan, jenis pengunjung, hingga apakah pengguna tersebut melakukan pembelian (Revenue). Dengan memanfaatkan teknologi Big Data seperti Apache Spark, analisis dapat dilakukan secara efisien terhadap data dalam skala besar.

Analisis ini bertujuan untuk menggali pola perilaku pengguna serta menemukan insight yang dapat digunakan untuk meningkatkan conversion rate dan strategi bisnis.

## Deskripsi
Dataset yang digunakan berisi informasi aktivitas pengunjung website, dengan atribut utama sebagai berikut:

- Administrative & Informational → jumlah halaman yang dikunjungi
- Durasi (Duration) → waktu yang dihabiskan pengguna
- ProductRelated → aktivitas terkait produk
- BounceRates & ExitRates → perilaku keluar dari website
- PageValues → nilai halaman terhadap potensi transaksi
- SpecialDay → kedekatan dengan hari spesial (promo/event)
- VisitorType → tipe pengunjung (New / Returning)
- Weekend → kunjungan di akhir pekan atau tidak
- Month → bulan kunjungan
- Revenue → apakah terjadi pembelian (target utama)

Dataset ini sangat cocok untuk analisis perilaku pengguna dan prediksi pembelian.

## Konteks Bisnis
Dalam konteks bisnis e-commerce, tujuan utama adalah:

🎯 Meningkatkan Conversion Rate

- Mengubah pengunjung menjadi pembeli adalah prioritas utama
- Dengan mengetahui faktor yang mempengaruhi pembelian,
- perusahaan bisa meningkatkan efektivitas strategi marketing.

🎯 Memahami Perilaku User

Perusahaan perlu mengetahui:

- Kapan user paling sering membeli
- Tipe user mana yang paling potensial
- Aktivitas apa yang menunjukkan niat membeli

🎯 Optimalisasi Strategi Marketing

Insight dari data dapat digunakan untuk:

- Menentukan waktu promo terbaik
- Personalisasi pengalaman pengguna
- Retargeting user yang belum membeli



# Setup & Load Data

In [ ]:
!pip install kagglehub pyspark

import kagglehub
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, countDistinct
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator, RegressionEvaluator

# 2. Download dataset
print("Mendownload dataset dari Kaggle...")
folder_path = kagglehub.dataset_download("thuandao/online-shoppers-purchasing-dataset")

# Cari nama file CSV yang ada di dalam folder tersebut
csv_filename = [f for f in os.listdir(folder_path) if f.endswith('.csv')][0]
file_path = os.path.join(folder_path, csv_filename)
print(f"Dataset berhasil didownload di: {file_path}")

# 3. Create SparkSession
spark = SparkSession.builder \
    .appName("OnlineShoppers_MLlib_P6") \
    .getOrCreate()
print("✅ SparkSession created!")

# 4. BACA DATA LANGSUNG MENGGUNAKAN SPARK (Sesuai Aturan Tugas)
# inferSchema=True membuat Spark otomatis mengenali mana kolom angka dan mana kolom teks
spark_df = spark.read.csv(file_path, header=True, inferSchema=True)

# 5. Cek Hasilnya
print(f"Shape: {spark_df.count()} baris, {len(spark_df.columns)} kolom")
spark_df.printSchema()

Mendownload dataset dari Kaggle...


100%|██████████| 54.5M/54.5M [00:00<00:00, 87.1MB/s]

Extracting files...


Dataset berhasil didownload di: /root/.cache/kagglehub/datasets/thuandao/online-shoppers-purchasing-dataset/versions/1/online_shoppers.csv
✅ SparkSession created!
Shape: 999999 baris, 18 kolom
root
 |-- Administrative: integer (nullable = true)
 |-- Administrative_Duration: double (nullable = true)
 |-- Informational: integer (nullable = true)
 |-- Informational_Duration: double (nullable = true)
 |-- ProductRelated: integer (nullable = true)
 |-- ProductRelated_Duration: double (nullable = true)
 |-- BounceRates: double (nullable = true)
 |-- ExitRates: double (nullable = true)
 |-- PageValues: double (nullable = true)
 |-- SpecialDay: double (nullable = true)
 |-- OperatingSystems: integer (nullable = true)
 |-- Browser: integer (nullable = true)
 |-- Region: integer (nullable = true)
 |-- TrafficType: integer (nullable = true)
 |-- Month: string (nullable = true)
 |-- VisitorType: string (nullable = true)
 |-- Weekend: boolean (nullable = true)
 |-- Revenue: boolean (nullable = true

# **Ekplorasi Data**



In [ ]:
print("=" * 80)
print("EKSPLORASI DATA MENGGUNAKAN SPARK")
print("=" * 80)

# 1. Info dasar
print("\n1️⃣ INFORMASI DASAR:")
print(f"Total baris: {spark_df.count()}")
print(f"Total kolom: {len(spark_df.columns)}")

# 2. Tipe data
print("\n2️⃣ TIPE DATA:")
spark_df.printSchema()

# 3. Cek missing values menggunakan Spark SQL
print("\n3️⃣ MISSING VALUES (per kolom):")
missing_summary = spark_df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in spark_df.columns]
)
missing_summary.show()

# 4. Statistik deskriptif
print("\n4️⃣ STATISTIK DESKRIPTIF:")
spark_df.describe().show()

# 5. Deteksi outlier dengan IQR (Interquartile Range)
print("\n5️⃣ DETEKSI OUTLIER (contoh: BounceRates):")
Q1 = spark_df.approxQuantile("BounceRates", [0.25], 0.01)[0]
Q3 = spark_df.approxQuantile("BounceRates", [0.75], 0.01)[0]
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.4f}, Q3: {Q3:.4f}, IQR: {IQR:.4f}")
print(f"Batas lower: {lower_bound:.4f}, Batas upper: {upper_bound:.4f}")

# 6. Cek duplikat
print("\n6️⃣ DUPLIKAT BARIS:")
duplicate_count = spark_df.count() - spark_df.dropDuplicates().count()
print(f"Jumlah duplikat: {duplicate_count}")

# 7. Analisis target variable (Revenue)
print("\n7️⃣ DISTRIBUSI TARGET VARIABLE (Revenue):")
spark_df.groupBy("Revenue").count().orderBy("Revenue").show()

EKSPLORASI DATA MENGGUNAKAN SPARK

1️⃣ INFORMASI DASAR:
Total baris: 999999
Total kolom: 18

2️⃣ TIPE DATA:
root
 |-- Administrative: integer (nullable = true)
 |-- Administrative_Duration: double (nullable = true)
 |-- Informational: integer (nullable = true)
 |-- Informational_Duration: double (nullable = true)
 |-- ProductRelated: integer (nullable = true)
 |-- ProductRelated_Duration: double (nullable = true)
 |-- BounceRates: double (nullable = true)
 |-- ExitRates: double (nullable = true)
 |-- PageValues: double (nullable = true)
 |-- SpecialDay: double (nullable = true)
 |-- OperatingSystems: integer (nullable = true)
 |-- Browser: integer (nullable = true)
 |-- Region: integer (nullable = true)
 |-- TrafficType: integer (nullable = true)
 |-- Month: string (nullable = true)
 |-- VisitorType: string (nullable = true)
 |-- Weekend: boolean (nullable = true)
 |-- Revenue: boolean (nullable = true)


3️⃣ MISSING VALUES (per kolom):
+--------------+-----------------------+---------

In [15]:
def detect_outliers_iqr(df, column_name):
    # Mengambil Q1 dan Q3 dengan akurasi 0.01 (1%)
    quantiles = df.approxQuantile(column_name, [0.25, 0.75], 0.01)
    q1, q3 = quantiles[0], quantiles[1]
    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    # Filter baris yang berada di luar batas
    outliers = df.filter((col(column_name) < lower_bound) | (col(column_name) > upper_bound))
    print(f"   - Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}")
    print(f"   - Range: {lower_bound:.2f} s/d {upper_bound:.2f}")
    print(f"   - Jumlah Outlier: {outliers.count()} baris")

detect_outliers_iqr(spark_df, "ProductRelated_Duration")

   - Q1: 0.00, Q3: 2407.47, IQR: 2407.47
   - Range: -3611.21 s/d 6018.69
   - Jumlah Outlier: 8325 baris


# **Data Cleaning & Prepare Data**

In [ ]:
print("\n" * 2)
print("=" * 80)
print("DATA CLEANING & PREPARATION")
print("=" * 80)

# 1. Hapus duplikat
print("\n1️⃣ HAPUS DUPLIKAT:")
spark_df_clean = spark_df.dropDuplicates()
print(f"Baris setelah hapus duplikat: {spark_df_clean.count()}")

# 2. Handle missing values (opsional, karena dataset ini tidak ada yang null)
print("\n2️⃣ MISSING VALUES HANDLING:")
print("Tidak ada missing values — dataset sudah bersih ✅")

# 3. Identifikasi fitur numerik dan kategorik
print("\n3️⃣ IDENTIFIKASI FITUR:")
numeric_cols = [f[0] for f in spark_df_clean.dtypes if f[1] in ['int', 'long', 'double', 'float']]
categorical_cols = [f[0] for f in spark_df_clean.dtypes if f[1] == 'string']
boolean_cols = [f[0] for f in spark_df_clean.dtypes if f[1] == 'boolean']

print(f"Numerik ({len(numeric_cols)}): {numeric_cols}")
print(f"Kategorik ({len(categorical_cols)}): {categorical_cols}")
print(f"Boolean ({len(boolean_cols)}): {boolean_cols}")

# 4. Target variable (Revenue)
print("\n4️⃣ TARGET VARIABLE:")
print("Target: 'Revenue' (klasifikasi: True/False)")

# 5. Features untuk model (exclude Revenue dan Month untuk sekarang)
features_for_model = [col for col in spark_df_clean.columns if col not in ['Revenue', 'Month', 'VisitorType']]
print(f"\nFeatures untuk model ({len(features_for_model)}): {features_for_model}")




DATA CLEANING & PREPARATION

1️⃣ HAPUS DUPLIKAT:
Baris setelah hapus duplikat: 999990

2️⃣ MISSING VALUES HANDLING:
Tidak ada missing values — dataset sudah bersih ✅

3️⃣ IDENTIFIKASI FITUR:
Numerik (14): ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']
Kategorik (2): ['Month', 'VisitorType']
Boolean (2): ['Weekend', 'Revenue']

4️⃣ TARGET VARIABLE:
Target: 'Revenue' (klasifikasi: True/False)

Features untuk model (15): ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend']


# **Feature Engineering**

## **Regresi**

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

# 1. FASE 2: Membuat Fitur Turunan (Derived Feature)
# Kita buat 'Total_Page_Views' untuk melihat total aktivitas user
df_reg = spark_df_clean.withColumn(
    "Total_Page_Views",
    col("Administrative") + col("Informational") + col("ProductRelated")
)



# 2. Menentukan Fitur & Target
# Target regresi: ProductRelated_Duration
target_col = "ProductRelated_Duration"
# Fitur: Semua kecuali target dan kolom kategori yang belum di-encode
features_reg_list = [
    'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration',
    'ProductRelated', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
    'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend', 'Total_Page_Views'
]

# 3. Pipeline Pre-processing Khusus Regresi
assembler_reg = VectorAssembler(inputCols=features_reg_list, outputCol="features_raw_reg")
scaler_reg = StandardScaler(inputCol="features_raw_reg", outputCol="features", withMean=True, withStd=True)
# Transform data
pipeline_prep = Pipeline(stages=[assembler_reg, scaler_reg])
df_reg_final = pipeline_prep.fit(df_reg).transform(df_reg)
# Rename target ke 'label' agar standar MLlib
df_reg_final = df_reg_final.withColumnRenamed(target_col, "label")
# 4. Split Data 70/20/10 (Sesuai ketentuan tugas)
train_reg, val_reg, test_reg = df_reg_final.randomSplit([0.7, 0.2, 0.1], seed=42)
print(f"✅ Data prepared for Regression with 1M rows.")
print(f"📊 Train: {train_reg.count()} | Val: {val_reg.count()} | Test: {test_reg.count()}")

## **Klasifikasi**

In [ ]:
print("\n" * 2)
print("=" * 80)
print("FEATURE ENGINEERING DENGAN SPARK MLLIB")
print("=" * 80)

# Define feature engineering stages
print("\n1️⃣ TAHAP 1: ENCODE CATEGORICAL FEATURES")

# StringIndexer untuk kategorik fitur
visitortype_indexer = StringIndexer(
    inputCol="VisitorType",
    outputCol="VisitorType_idx",
    handleInvalid="keep"
)

month_indexer = StringIndexer(
    inputCol="Month",
    outputCol="Month_idx",
    handleInvalid="keep"
)

# Label encoding untuk target (Revenue)
label_indexer = StringIndexer(
    inputCol="Revenue",
    outputCol="label",
    handleInvalid="skip"
)

print("✅ StringIndexer defined untuk: VisitorType, Month, Revenue")

# ========================================
print("\n2️⃣ TAHAP 2: VECTOR ASSEMBLY")

# Gabungkan semua numerik features + indexed categorical
features_to_assemble = [
    'Administrative',
    'Administrative_Duration',
    'Informational',
    'Informational_Duration',
    'ProductRelated',
    'ProductRelated_Duration',
    'BounceRates',
    'ExitRates',
    'PageValues',
    'SpecialDay',
    'OperatingSystems',
    'Browser',
    'Region',
    'TrafficType',
    'Weekend',
    'VisitorType_idx',
    'Month_idx'
]

assembler = VectorAssembler(
    inputCols=features_to_assemble,
    outputCol="features_raw",
    handleInvalid="keep"
)

print(f"✅ VectorAssembler akan menggabungkan {len(features_to_assemble)} fitur")

# ========================================
print("\n3️⃣ TAHAP 3: NORMALISASI (STANDARDIZATION)")

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

print("✅ StandardScaler akan menormalkan skala fitur")

# ========================================
print("\n4️⃣ BANGUN PIPELINE FEATURE ENGINEERING")

feature_engineering_pipeline = Pipeline(stages=[
    visitortype_indexer,
    month_indexer,
    label_indexer,
    assembler,
    scaler
])

print("✅ Pipeline feature engineering berhasil dibangun")

# Ubah kolom tipe boolean menjadi string
spark_df_clean = spark_df_clean.withColumn("Revenue", col("Revenue").cast("string"))
spark_df_clean = spark_df_clean.withColumn("Weekend", col("Weekend").cast("integer"))

# ========================================
print("\n5️⃣ JALANKAN PIPELINE PADA DATA")

# Fit + transform
df_processed = feature_engineering_pipeline.fit(spark_df_clean).transform(spark_df_clean)

print(f"✅ Pipeline dijalankan pada {df_processed.count()} baris data")

# Verifikasi output
print("\n6️⃣ VERIFIKASI OUTPUT PIPELINE:")
df_processed.select(
    "Administrative",
    "VisitorType",
    "VisitorType_idx",
    "features",
    "label"
).show(5, truncate=False)

print("\nSchema output pipeline:")
df_processed.select("features", "label").printSchema()




FEATURE ENGINEERING DENGAN SPARK MLLIB

1️⃣ TAHAP 1: ENCODE CATEGORICAL FEATURES
✅ StringIndexer defined untuk: VisitorType, Month, Revenue

2️⃣ TAHAP 2: VECTOR ASSEMBLY
✅ VectorAssembler akan menggabungkan 17 fitur

3️⃣ TAHAP 3: NORMALISASI (STANDARDIZATION)
✅ StandardScaler akan menormalkan skala fitur

4️⃣ BANGUN PIPELINE FEATURE ENGINEERING
✅ Pipeline feature engineering berhasil dibangun

5️⃣ JALANKAN PIPELINE PADA DATA
✅ Pipeline dijalankan pada 999990 baris data

6️⃣ VERIFIKASI OUTPUT PIPELINE:
+--------------+-----------------+---------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|Administrative|VisitorType      |VisitorType_idx|features                                 

# **Train-Test Split**



In [ ]:
print("\n" * 2)
print("=" * 80)
print("STEP 4: TRAIN-TEST SPLIT")
print("=" * 80)

# Split data: 70% train, 20% validation, 10% test
train_df, val_df, test_df = df_processed.randomSplit([0.7, 0.2, 0.1], seed=42)

print(f"\n📊 DATA SPLIT:")
print(f"  Training set  : {train_df.count()} baris ({train_df.count()/df_processed.count()*100:.1f}%)")
print(f"  Validation set: {val_df.count()} baris ({val_df.count()/df_processed.count()*100:.1f}%)")
print(f"  Test set      : {test_df.count()} baris ({test_df.count()/df_processed.count()*100:.1f}%)")

# Cek class balance di setiap set
print(f"\n📊 CLASS DISTRIBUTION DALAM SETIAP SET:")

print("\nTraining set:")
train_df.groupBy("label").count().show()

print("Validation set:")
val_df.groupBy("label").count().show()

print("Test set:")
test_df.groupBy("label").count().show()




STEP 4: TRAIN-TEST SPLIT

📊 DATA SPLIT:
  Training set  : 699967 baris (70.0%)
  Validation set: 199749 baris (20.0%)
  Test set      : 100274 baris (10.0%)

📊 CLASS DISTRIBUTION DALAM SETIAP SET:

Training set:
+-----+------+
|label| count|
+-----+------+
|  0.0|591585|
|  1.0|108382|
+-----+------+

Validation set:
+-----+------+
|label| count|
+-----+------+
|  0.0|168949|
|  1.0| 30800|
+-----+------+

Test set:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|84712|
|  1.0|15562|
+-----+-----+



# **Klasifikasi**:     
Klasifikasi digunakan untuk menjawab petanyaan:     
***Bagaimana kita bisa memprediksi secara akurat apakah seorang pengunjung website akan melakukan transaksi (Revenue) berdasarkan pola perilaku mereka (seperti PageValues, BounceRates, dan VisitorType)?***

Intinya: Klasifikasi apakah revenue bernilai True atau False

## **1. Random Forest Classifier**

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.classification import RandomForestClassifier
from pyspark.sql.functions import col

print("=" * 80)
print("STEP 5: MODEL 1 - RANDOM FOREST CLASSIFIER")
print("=" * 80)

# A. PERSIAPAN JURI (EVALUATORS)
# Kita siapkan juri sebelum menguji model
print("Menyiapkan evaluator metrik...")
eval_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
eval_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
eval_prec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision")
eval_rec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall")
eval_f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

# B. MEMBANGUN DAN MELATIH MODEL (TRAINING)
print("\n1️⃣ MEMBANGUN DAN MELATIH MODEL RANDOM FOREST...")
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=3,
    seed=42
)

# Latih model menggunakan data training
rf_model = rf.fit(train_df)
print("✅ Training selesai!")

# C. PREDIKSI DATA VALIDATION
print("\n2️⃣ MELAKUKAN PREDIKSI PADA DATA VALIDATION...")
# Uji model menggunakan data validasi
rf_val_predictions = rf_model.transform(val_df)

# D. EVALUASI SEMUA METRIK (Sesuai Syarat Tugas)
print("\n3️⃣ EVALUASI METRIK PADA DATA VALIDATION:")
auc_score = eval_auc.evaluate(rf_val_predictions)
acc_score = eval_acc.evaluate(rf_val_predictions)
prec_score = eval_prec.evaluate(rf_val_predictions)
rec_score = eval_rec.evaluate(rf_val_predictions)
f1_score = eval_f1.evaluate(rf_val_predictions)

print(f" AUC       : {auc_score:.4f}")
print(f" Accuracy  : {acc_score:.4f}")
print(f" Precision : {prec_score:.4f}")
print(f" Recall    : {rec_score:.4f}")
print(f" F1-Score  : {f1_score:.4f}")

# E. CONFUSION MATRIX MANUAL (Syarat Wajib Dosen)
print("\n4️⃣ CONFUSION MATRIX (VALIDATION SET):")

# TP: Benar-benar memprediksi 1 (Beli)
tp = rf_val_predictions.filter((col("label") == 1.0) & (col("prediction") == 1.0)).count()
# FP: Salah memprediksi 1 (Ditebak Beli, aslinya Tidak)
fp = rf_val_predictions.filter((col("label") == 0.0) & (col("prediction") == 1.0)).count()
# TN: Benar-benar memprediksi 0 (Tidak Beli)
tn = rf_val_predictions.filter((col("label") == 0.0) & (col("prediction") == 0.0)).count()
# FN: Salah memprediksi 0 (Ditebak Tidak beli, aslinya Beli)
fn = rf_val_predictions.filter((col("label") == 1.0) & (col("prediction") == 0.0)).count()

print(f"- True Positives  (TP) : {tp} ")
print(f"- False Positives (FP) : {fp} ")
print(f"- True Negatives  (TN) : {tn} ")
print(f"- False Negatives (FN) : {fn} ")

# F. FEATURE IMPORTANCE (Untuk Analisis Bisnis Fase 4)
print("\n5️⃣ TOP 5 FITUR PALING BERPENGARUH (FEATURE IMPORTANCE):")
# Ini daftar nama kolom persis seperti yang kamu buat di Step 2 VectorAssembler
feature_cols = [
    'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues',
    'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend',
    'VisitorType_idx', 'Month_idx'
]

importances = rf_model.featureImportances.toArray()

# Gabungkan nama kolom dengan nilai bobotnya, lalu urutkan dari terbesar ke terkecil
feature_importance_list = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)

# Tampilkan 5 teratas
for i, (feat, imp) in enumerate(feature_importance_list[:5]):
    print(f" {i+1}. {feat:25s}: {imp:.4f}")

STEP 5: MODEL 1 - RANDOM FOREST CLASSIFIER
Menyiapkan evaluator metrik...

1️⃣ MEMBANGUN DAN MELATIH MODEL RANDOM FOREST...
✅ Training selesai!

2️⃣ MELAKUKAN PREDIKSI PADA DATA VALIDATION...

3️⃣ EVALUASI METRIK PADA DATA VALIDATION:
 AUC       : 0.9393
 Accuracy  : 0.9248
 Precision : 0.9251
 Recall    : 0.9248
 F1-Score  : 0.9166

4️⃣ CONFUSION MATRIX (VALIDATION SET):
- True Positives  (TP) : 17074 
- False Positives (FP) : 1302 
- True Negatives  (TN) : 167647 
- False Negatives (FN) : 13726 

5️⃣ TOP 5 FITUR PALING BERPENGARUH (FEATURE IMPORTANCE):
 1. PageValues               : 0.5721
 2. ExitRates                : 0.1814
 3. BounceRates              : 0.1139
 4. ProductRelated           : 0.0541
 5. ProductRelated_Duration  : 0.0363


# **Regresi**:     
Regresi digunakan untuk menjawab petanyaan:     
***Berapa estimasi waktu yang akan dihabiskan pengunjung di halaman terkait produk (ProductRelated_Duration) berdasarkan karakteristik sesi mereka dan jenis halaman lain yang mereka buka?***

Intinya: Regresi nilai `ProductRelated_Duration`

## **1. Linear Reggresion**

In [18]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

print("=" * 80)
print("LINEAR REGRESSION")
print("=" * 80)

# Inisialisasi Linear Regression
lr = LinearRegression(featuresCol="features", labelCol="label", maxIter=100, regParam=0.1)

# Fit Model
lr_model = lr.fit(train_reg)

# Prediksi pada Validation Set
lr_predictions = lr_model.transform(val_reg)

# Evaluasi Metrik
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction")
rmse_lr = evaluator.evaluate(lr_predictions, {evaluator.metricName: "rmse"})
mae_lr = evaluator.evaluate(lr_predictions, {evaluator.metricName: "mae"})
r2_lr = evaluator.evaluate(lr_predictions, {evaluator.metricName: "r2"})

print(f"🎯 Linear Regression Metrics:")
print(f"   - RMSE: {rmse_lr:.4f}")
print(f"   - MAE : {mae_lr:.4f}")
print(f"   - R2  : {r2_lr:.4f}")

print("\n💡 TOP 5 INFLUENTIAL FEATURES:")
# For Linear Regression, we use coefficients to determine feature influence
# Take the absolute value to represent 'importance' regardless of direction
importances = abs(lr_model.coefficients.toArray())
for i, (feat, imp) in enumerate(sorted(zip(features_reg_list, importances), key=lambda x: x[1], reverse=True)[:5]):
    print(f" {i+1}. {feat:25s}: {imp:.4f}")

LINEAR REGRESSION
🎯 Linear Regression Metrics:
   - RMSE: 765.8841
   - MAE : 574.9505
   - R2  : 0.7405

💡 TOP 5 INFLUENTIAL FEATURES:
 1. ProductRelated           : 651.2507
 2. Total_Page_Views         : 613.2607
 3. Administrative_Duration  : 199.7710
 4. Administrative           : 165.9024
 5. Informational_Duration   : 133.6212


# **2. Random Forest Regressor**

In [12]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

print("=" * 80)
print("RANDOM FOREST REGRESSOR (WITH TUNING)")
print("=" * 80)

# Inisialisasi RF
rf = RandomForestRegressor(featuresCol="features", labelCol="label", seed=42)

# Buat Parameter Grid (Tuning Hyperparameter)
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [20, 50]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

# CrossValidator untuk Tuning Otomatis
cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3
)

# Fit CV Model
print("GASS TUNINGGGGGG!!!")
cv_model = cv.fit(train_reg)

# Ambil Model Terbaik
best_rf_model = cv_model.bestModel

# Prediksi pada Validation Set
rf_predictions = best_rf_model.transform(val_reg)

# Evaluasi Metrik
rmse_rf = evaluator.evaluate(rf_predictions, {evaluator.metricName: "rmse"})
mae_rf = evaluator.evaluate(rf_predictions, {evaluator.metricName: "mae"})
r2_rf = evaluator.evaluate(rf_predictions, {evaluator.metricName: "r2"})

print(f"🎯 Random Forest (Best Model) Metrics:")
print(f"   - RMSE: {rmse_rf:.4f}")
print(f"   - MAE : {mae_rf:.4f}")
print(f"   - R2  : {r2_rf:.4f}")

# Feature Importance untuk Fase 4
print("\n💡 TOP 5 INFLUENTIAL FEATURES:")
importances = best_rf_model.featureImportances.toArray()
for i, (feat, imp) in enumerate(sorted(zip(features_reg_list, importances), key=lambda x: x[1], reverse=True)[:5]):
    print(f" {i+1}. {feat:25s}: {imp:.4f}")

RANDOM FOREST REGRESSOR (WITH TUNING)
GASS TUNINGGGGGG!!!
🎯 Random Forest (Best Model) Metrics:
   - RMSE: 770.2572
   - MAE : 580.2241
   - R2  : 0.7375

💡 TOP 5 INFLUENTIAL FEATURES:
 1. ProductRelated           : 0.5280
 2. Total_Page_Views         : 0.3920
 3. Informational_Duration   : 0.0219
 4. Informational            : 0.0210
 5. Administrative_Duration  : 0.0181


## **3. GBT Regressor**

In [13]:
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

print("\n" * 2)
print("=" * 80)
print("GBT REGRESSOR")
print("=" * 80)

# A. PERSIAPAN DATA UNTUK REGRESI
print("\n1️⃣ MEMPERSIAPKAN DATA UNTUK MODEL REGRESI (ProductRelated_Duration sebagai target)...")

# 1. Definisi StringIndexer untuk fitur kategorikal
# Re-use existing StringIndexer definitions if they are consistent
visitortype_indexer_reg = StringIndexer(
    inputCol="VisitorType",
    outputCol="VisitorType_idx",
    handleInvalid="keep"
)
month_indexer_reg = StringIndexer(
    inputCol="Month",
    outputCol="Month_idx",
    handleInvalid="keep"
)

# 2. Fitur yang akan digunakan sebagai input (features) untuk regresi
# 'ProductRelated_Duration' akan menjadi target (label), jadi jangan dimasukkan ke features.
# 'Revenue' juga tidak dipakai sebagai feature untuk memprediksi durasi.
features_for_regression_assembly = [
    'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration',
    'ProductRelated', 'BounceRates', 'ExitRates', 'PageValues',
    'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend',
    'VisitorType_idx', 'Month_idx'
]

assembler_reg = VectorAssembler(
    inputCols=features_for_regression_assembly,
    outputCol="features_raw_reg",
    handleInvalid="keep"
)

# 3. Normalisasi Fitur
scaler_reg = StandardScaler(
    inputCol="features_raw_reg",
    outputCol="features",
    withMean=True,
    withStd=True
)

# 4. Bangun Pipeline Feature Engineering untuk Regresi
regression_feature_pipeline = Pipeline(stages=[
    visitortype_indexer_reg,
    month_indexer_reg,
    assembler_reg,
    scaler_reg
])

# Pastikan spark_df_clean memiliki 'Weekend' sebagai integer
# Ini sudah dilakukan di sel sebelumnya, tapi bisa diulang untuk memastikan
spark_df_clean_reg = spark_df_clean.withColumn("Weekend", col("Weekend").cast("integer"))

# Jalankan pipeline untuk mendapatkan data yang siap untuk regresi
df_for_gbt = regression_feature_pipeline.fit(spark_df_clean_reg).transform(spark_df_clean_reg)

# Ganti nama kolom target 'ProductRelated_Duration' menjadi 'label' sesuai konvensi MLlib
df_for_gbt = df_for_gbt.withColumnRenamed("ProductRelated_Duration", "label_reg")

print("✅ Data siap untuk GBT Regressor.")

# B. TRAIN-TEST SPLIT UNTUK REGRESI
print("\n2️⃣ MEMBAGI DATA MENJADI TRAINING DAN TEST SET...")
train_gbt, test_gbt = df_for_gbt.randomSplit([0.8, 0.2], seed=42)
print(f"  Training set: {train_gbt.count()} baris")
print(f"  Test set    : {test_gbt.count()} baris")

# C. MEMBANGUN DAN MELATIH MODEL GBT REGRESSOR
print("\n3️⃣ MEMBANGUN DAN MELATIH MODEL GBT REGRESSOR...")
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="label_reg",
    maxIter=10, # Jumlah iterasi/pohon
    maxDepth=5, # Kedalaman maksimum pohon
    seed=42
)

# Latih model menggunakan data training
gbt_model = gbt.fit(train_gbt)
print("✅ Training GBT Regressor selesai!")

# D. PREDIKSI DATA TEST
print("\n4️⃣ MELAKUKAN PREDIKSI PADA DATA TEST...")
gbt_predictions = gbt_model.transform(test_gbt)

# E. EVALUASI MODEL REGRESI
print("\n5️⃣ EVALUASI METRIK REGRESI PADA DATA TEST:")

evaluator_rmse = RegressionEvaluator(labelCol="label_reg", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(gbt_predictions)

evaluator_mae = RegressionEvaluator(labelCol="label_reg", predictionCol="prediction", metricName="mae")
mae = evaluator_mae.evaluate(gbt_predictions)

evaluator_r2 = RegressionEvaluator(labelCol="label_reg", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(gbt_predictions)

print(f" 🎯 Root Mean Squared Error (RMSE) : {rmse:.4f}")
print(f" 🎯 Mean Absolute Error (MAE)      : {mae:.4f}")
print(f" 🎯 R-squared (R2)                 : {r2:.4f}")

# F. FEATURE IMPORTANCE
print("\n6️⃣ TOP 5 FITUR PALING BERPENGARUH (FEATURE IMPORTANCE) PADA GBT REGRESSOR:")
importances_gbt = gbt_model.featureImportances.toArray()

feature_importance_list_gbt = sorted(zip(features_for_regression_assembly, importances_gbt), key=lambda x: x[1], reverse=True)

for i, (feat, imp) in enumerate(feature_importance_list_gbt[:5]):
    print(f" {i+1}. {feat:25s}: {imp:.4f}")




GBT REGRESSOR

1️⃣ MEMPERSIAPKAN DATA UNTUK MODEL REGRESI (ProductRelated_Duration sebagai target)...
✅ Data siap untuk GBT Regressor.

2️⃣ MEMBAGI DATA MENJADI TRAINING DAN TEST SET...
  Training set: 799844 baris
  Test set    : 200146 baris

3️⃣ MEMBANGUN DAN MELATIH MODEL GBT REGRESSOR...
✅ Training GBT Regressor selesai!

4️⃣ MELAKUKAN PREDIKSI PADA DATA TEST...

5️⃣ EVALUASI METRIK REGRESI PADA DATA TEST:
 🎯 Root Mean Squared Error (RMSE) : 780.0309
 🎯 Mean Absolute Error (MAE)      : 584.6049
 🎯 R-squared (R2)                 : 0.7319

6️⃣ TOP 5 FITUR PALING BERPENGARUH (FEATURE IMPORTANCE) PADA GBT REGRESSOR:
 1. ProductRelated           : 0.8803
 2. Administrative_Duration  : 0.0467
 3. Informational_Duration   : 0.0374
 4. Administrative           : 0.0331
 5. PageValues               : 0.0015
